# Federated FinDiff with Differential Privacy on UCI Credit Card data

## Goal

This notebook is a guided implementation of DP-FedTabDiff. It connects the paper's mathematical ideas to executable PyTorch code: mixed-type records are transformed into a continuous representation, a diffusion model learns to predict injected noise, several clients train locally, and a server aggregates their updates.

The goal is understanding rather than benchmark reproduction. The experiment is intentionally small enough for a CPU and downloads UCI Credit Card dataset **ID 350** at runtime; no real data is committed to the repository.

## Main content

The walkthrough covers data inspection, leakage-safe preprocessing, FinDiff representation learning, local DP-SGD, target-epsilon accounting with Opacus, FedAvg aggregation, held-out evaluation, reverse diffusion sampling, generated synthetic data plots, and SDV quality metrics.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from ucimlrepo import fetch_ucirepo
from sdv.metadata import SingleTableMetadata
from sdmetrics.reports.single_table import QualityReport

from dp_fedtabdiff.data import DataTransformer, iid_partition
from dp_fedtabdiff.diffusion import GaussianDiffusion
from dp_fedtabdiff.federated import fedavg
from dp_fedtabdiff.findiff import FinDiff, FinDiffPrivateWrapper, findiff_loss
from dp_fedtabdiff.privacy import PersistentPrivacyEngine

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cpu")

## 1. Acquire and inspect the dataset

This section shows the raw feature table and target distribution before modeling. We normalize names, remove incomplete rows, and keep the binary default target for conditional diffusion. The target remains separate so it can be used explicitly as conditioning information.

In [ ]:
# fetch dataset
default_of_credit_card_clients = fetch_ucirepo(id=350)

# data (as pandas dataframes)
X = default_of_credit_card_clients.data.features
y = default_of_credit_card_clients.data.targets

# map X column names to variable names
cols = default_of_credit_card_clients.variables
X.columns = cols[cols["name"].isin(X.columns)]["description"].to_list()
frame = X.copy()
target = y.copy()

display(frame.head())
print(frame.shape, target.value_counts().to_dict())

In [ ]:
# uci = fetch_ucirepo(id=350)
# frame = uci.data.features.copy()
# target = uci.data.targets.iloc[:, 0].rename("default")
# frame.columns = [str(column).strip().lower().replace(" ", "_") for column in frame.columns]
# valid = ~frame.isna().any(axis=1)
# frame = frame.loc[valid].reset_index(drop=True)
# target = target.loc[valid].astype(int).reset_index(drop=True)
# display(frame.head())
# print(frame.shape, target.value_counts().to_dict())

## 2. Fit mixed-type preprocessing on the training set only

Low-cardinality attributes are treated as categorical, while financial amounts remain numerical. The transformer learns vocabularies and scaling parameters using only the training split, then applies them unchanged to the test split. This prevents evaluation statistics from leaking into the representation. Unseen categories use an explicit unknown token.

In [ ]:
train_frame, test_frame, train_target, test_target = train_test_split(
    frame, target, test_size=0.2, random_state=SEED, stratify=target
)
categorical_columns = [column for column in frame.columns if frame[column].nunique() <= 12]
numerical_columns = [column for column in frame.columns if column not in categorical_columns]
transformer = DataTransformer(categorical_columns, numerical_columns, numerical_scaler="quantile")
train_encoded = transformer.fit_transform(train_frame)
test_encoded = transformer.transform(test_frame)
category_sizes = [len(transformer.category_encoder.vocabularies[column]) - 1 for column in categorical_columns]
print("categorical columns:", categorical_columns)
print("numerical columns:", numerical_columns)
print("category sizes:", category_sizes)

## 3. Understand FinDiff's representation

Unlike a plain tabular MLP, FinDiff uses an embedding table with disjoint token ranges for each categorical column. The embedded categorical values are flattened and concatenated with scaled numerical values to form continuous `x_0`. The denoiser receives noisy `x_t`, a sinusoidal timestep embedding, and either a label embedding or a learned unconditional embedding. The default residual MLP, plus Transformer support and distance/logit decoding, follows the original FinDiff implementation.

In [ ]:
train_cat = torch.tensor(train_encoded["cat"], dtype=torch.long)
train_num = torch.tensor(train_encoded["num"], dtype=torch.float32)
test_cat = torch.tensor(test_encoded["cat"], dtype=torch.long)
test_num = torch.tensor(test_encoded["num"], dtype=torch.float32)
train_labels = torch.tensor(train_target.to_numpy(), dtype=torch.long)
test_labels = torch.tensor(test_target.to_numpy(), dtype=torch.long)
template = FinDiff(category_sizes, train_num.shape[1], embedding_dim=2, hidden_dims=(128, 128), condition_dim=64, num_labels=2)
print("FinDiff continuous dimension:", template.data_dim)
print("encoded training rows:", template.encode(train_cat[:2], train_num[:2]).shape)

## 4. Simulate clients and aggregate on the server

The partitions are simulated in one process, but the data flow mirrors a federated deployment. Each client owns private data, a local optimizer, a FinDiff model, and a persistent privacy accountant. It receives global weights, performs a local DP-SGD update, and returns weights plus its sample count; the server applies sample-weighted FedAvg. `make_private_with_epsilon` treats the configured epsilon as a target and calibrates the noise multiplier for the planned training schedule. Achieved epsilon is measured after each round.

In [ ]:
diffusion = GaussianDiffusion(steps=100).to(device)
global_model = FinDiff(category_sizes, train_num.shape[1], embedding_dim=2, hidden_dims=(128, 128), condition_dim=64, num_labels=2, embedding_learned=False)
client_indices = iid_partition(len(train_cat), n_clients=3, seed=SEED)

# Each client owns one model, optimizer, data loader, and accountant for the
# entire experiment. Re-attaching Opacus every round would reset epsilon.
dp_config = {"target_epsilon": 3.0, "target_delta": 1e-5, "max_grad_norm": 1.0, "batch_size": 256, "epochs": 3}
clients = []
for indices in client_indices:
    client = FinDiff(category_sizes, train_num.shape[1], embedding_dim=2, hidden_dims=(128, 128), condition_dim=64, num_labels=2, embedding_learned=False)
    for name, parameter in client.named_parameters():
        if not name.startswith("backbone."):
            parameter.requires_grad_(False)
    private_model = FinDiffPrivateWrapper(client, diffusion)
    optimizer = torch.optim.Adam(private_model.parameters(), lr=1e-3)
    packed = torch.cat((train_cat[indices].float(), train_num[indices], train_labels[indices].float()), dim=1)
    loader = DataLoader(TensorDataset(packed), batch_size=dp_config["batch_size"], shuffle=True)
    privacy = PersistentPrivacyEngine(dp_config["target_delta"], None, dp_config["max_grad_norm"], target_epsilon=dp_config["target_epsilon"], epochs=dp_config["epochs"])
    private_model, optimizer, loader = privacy.attach(private_model, optimizer, loader)
    clients.append({"model": private_model, "optimizer": optimizer, "loader": loader, "privacy": privacy, "size": len(indices)})

round_losses, round_epsilons = [], []
for communication_round in range(3):
    client_states, client_sizes, losses = [], [], []
    for state in clients:
        state["model"]._module.model.load_state_dict(global_model.state_dict())
        state["model"].train()
        (packed_batch,) = next(iter(state["loader"]))
        state["optimizer"].zero_grad(set_to_none=True)
        loss = state["model"](packed_batch).mean()
        loss.backward()
        state["optimizer"].step()
        losses.append(float(loss.detach()))
        client_states.append(state["model"]._module.model.state_dict())
        client_sizes.append(state["size"])
    global_model.load_state_dict(fedavg(client_states, client_sizes))
    global_model.eval()
    with torch.no_grad():
        utility_loss = float(findiff_loss(global_model, diffusion, test_cat, test_num, test_labels))
    round_losses.append(utility_loss)
    round_epsilons.append(float(np.mean([state["privacy"].epsilon() for state in clients])))
    print(f"round {communication_round + 1}: client losses={np.round(losses, 4)}, test MSE={utility_loss:.4f}, mean epsilon={round_epsilons[-1]:.3f} / target {dp_config['target_epsilon']:.1f}")

## 5. Evaluate the aggregated model on the held-out test set

We measure noise-prediction MSE on unseen rows after each aggregation. This is a generalization diagnostic, not a complete synthetic-data utility score. The paired plot shows how held-out utility changes as the mean client epsilon approaches the requested target.

In [ ]:
global_model.eval()
test_loss = round_losses[-1]
print("held-out test noise-prediction MSE:", round(test_loss, 4))
fig, ax1 = plt.subplots(figsize=(6, 3.5))
ax1.plot(range(1, len(round_losses) + 1), round_losses, marker="o", color="tab:blue")
ax1.set_xlabel("communication round")
ax1.set_ylabel("test noise-prediction MSE", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(range(1, len(round_epsilons) + 1), round_epsilons, marker="s", color="tab:red", label="achieved epsilon")
ax2.axhline(dp_config["target_epsilon"], color="tab:red", linestyle="--", alpha=0.6, label="target epsilon")
ax2.set_ylabel("mean client epsilon", color="tab:red")
ax2.legend(loc="upper right")
ax1.set_title("Privacy budget versus held-out utility")
fig.tight_layout()
plt.show()

## 6. Sample synthetic rows from the global FinDiff model

Sampling starts from Gaussian noise and applies the learned reverse process from the final timestep back to zero. Held-out labels are used as a transparent conditional-generation example; an application could choose a desired class balance instead. Categorical coordinates are decoded by nearest learned embedding and numerical coordinates are inverse-transformed with the training-only transformer.

In [ ]:
n_samples = min(1000, len(test_frame))
sample_labels = test_labels[:n_samples]
z = torch.randn(n_samples, global_model.data_dim)
with torch.no_grad():
    for step in reversed(range(diffusion.steps)):
        t = torch.full((n_samples,), step, dtype=torch.long)
        z = diffusion.predict_mean(z, global_model(z, t, sample_labels), t)
        if step > 0:
            z = z + diffusion.betas[step].sqrt() * torch.randn_like(z)
sampled_cat = global_model.decode_categories(z).numpy()
sampled_num = z[:, len(categorical_columns) * global_model.embedding_dim:].numpy()
synthetic = transformer.inverse_transform(sampled_cat, sampled_num)
synthetic["default"] = sample_labels.numpy()
synthetic.head()

## 7. SDV quality report and visual diagnostics

SDV compares column distributions and pairwise relationships between held-out real data and generated data. KDE plots provide a smooth visual comparison of numerical densities. These are fidelity and utility diagnostics, not privacy guarantees; formal privacy is represented by Opacus epsilon and delta. The held-out test set is the real reference, avoiding comparison with the local training rows.

In [ ]:
real_for_sdv = test_frame.copy()
real_for_sdv["default"] = test_target.to_numpy()
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(real_for_sdv)
quality_report = QualityReport()
quality_report.generate(real_for_sdv, synthetic, metadata=metadata.to_dict())
print("SDV quality score:", quality_report.get_score())
display(quality_report.get_properties())
display(quality_report.get_details(property_name="Column Shapes").head())

In [ ]:
for column in numerical_columns[:3]:
    plt.figure(figsize=(5, 3))
    sns.kdeplot(real_for_sdv[column], fill=True, alpha=0.35, label="real test")
    sns.kdeplot(synthetic[column], fill=True, alpha=0.35, label="synthetic")
    plt.title(f"KDE: {column}")
    plt.xlabel(column)
    plt.legend()
    plt.tight_layout()
    plt.show()
print("client partitions:", [len(indices) for indices in client_indices])

## Final remarks and summary

This experiment demonstrates the complete conceptual chain:

`raw mixed-type table → train-only encoding → FinDiff x_0 → noisy x_t → local DP denoising → FedAvg global model → reverse diffusion → decoded synthetic table`.

The example is intentionally educational. Three IID clients, a few communication rounds, a small residual backbone, and a short diffusion schedule are not sufficient to reproduce paper-scale results. For research experiments, increase the data, model, rounds, and evaluation repetitions using the configuration files in the repository.
